# 04: replicating the Bernstein–Lagarias tables

Companion to *A Bit Shift Wearing A Disguise* (`paper/`). **[ESTABLISHED:CALCULATED]** for the agreements, **[OPEN:CALCULATED]** for the one disagreement. This notebook is a code paper's notebook: it runs `engine/e_collatz_bl_replication.py` and shows what comes out. It proves nothing.

The conjugacy map Φ = Q⁻¹ of Bernstein & Lagarias, *The 3x+1 conjugacy map*, Canad. J. Math. 48 (1996), 1154–1169 (copy in `references/`), induces permutations of Z/2ⁿ. Their Tables 2.2, 2.3 and 2.4 count cycles; the engine re-computes them (the printed values are transcribed into the engine, with page numbers).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / 'engine'))   # run this notebook from CollatzShift/
import e_collatz_bl_replication as bl
res = bl.run(verbose=True, n_cycles=16, n_fixed=100)

Bernstein-Lagarias (1996) replication (ESTABLISHED:CALCULATED (Bernstein-Lagarias 1996, re-computed), with ONE RECORDED DISCREPANCY WINDOW in the printed tables (see mismatch lists))
  Table 2.2 (cycle lengths on the odd residues): rows n=2..16 compared, mismatches: none
  n   cycles by length (1,2,4,...)                      order
   2  [2]                                                  1
   3  [2, 1]                                               2
   4  [4, 2]                                               2
   5  [6, 3, 1]                                            4
   6  [6, 7, 3]                                            4
   7  [8, 10, 3, 3]                                        8
   8  [14, 17, 8, 0, 3]                                    16
   9  [14, 21, 18, 4, 0, 3]                                32
  10  [10, 35, 24, 14, 2, 0, 3]                            64
  11  [12, 40, 37, 18, 12, 2, 0, 3]                        128
  12  [16, 48, 70, 23, 16, 10, 2, 0, 3]            

## The cycle table (their Table 2.2), as the engine sees it

`odd_cycle_counts(n)` returns, for each n, how many cycles of length 1, 2, 4, 8, … the permutation has on the odd residues. Every cycle length is a power of two (the function raises if not).

In [2]:
for n in (5, 8, 12):
    print(n, bl.odd_cycle_counts(n), ' printed Table 2.2:', bl.TABLE_2_2[n])

5 [6, 3, 1]  printed Table 2.2: [6, 3, 1]
8 [14, 17, 8, 0, 3]  printed Table 2.2: [14, 17, 8, 0, 3]
12 [16, 48, 70, 23, 16, 10, 2, 0, 3]  printed Table 2.2: [16, 48, 70, 23, 16, 10, 2, 0, 3]


## Two algorithms for the odd fixed points, and the disagreement

`odd_fixed_point_counts` extends odd fixed points one bit at a time; `odd_fixed_points_dfs` is an independent bit-by-bit search that tracks the affine form of Tʲ. They agree with each other. Against the printed tables they agree everywhere **except** a block, where the printed value for n equals the computed value for n+1.

In [3]:
dfs = bl.odd_fixed_points_dfs(1000)
q = bl.odd_fixed_point_counts(100)
print('two algorithms agree for n<=100:', all(dfs[n] == q[n] for n in range(1, 101)))
print('  n  computed  printed(2.3)  computed(n+1)')
for n in range(63, 82):
    mark = '' if dfs[n] == bl.TABLE_2_3[n] else '   <-- differs; printed == computed(n+1): %s' % (bl.TABLE_2_3[n] == dfs[n + 1])
    print(f'{n:3d}  {dfs[n]:7d}  {bl.TABLE_2_3[n]:11d}  {dfs[n+1]:12d}{mark}')

two algorithms agree for n<=100: True
  n  computed  printed(2.3)  computed(n+1)
 63      110          110           112
 64      112          112           106
 65      106          106           102
 66      102           92            92   <-- differs; printed == computed(n+1): True
 67       92          106           106   <-- differs; printed == computed(n+1): True
 68      106          116           116   <-- differs; printed == computed(n+1): True
 69      116          122           122   <-- differs; printed == computed(n+1): True
 70      122          132           132   <-- differs; printed == computed(n+1): True
 71      132          152           152   <-- differs; printed == computed(n+1): True
 72      152          144           144   <-- differs; printed == computed(n+1): True
 73      144          120           120   <-- differs; printed == computed(n+1): True
 74      120          108           108   <-- differs; printed == computed(n+1): True
 75      108          114

**We do not know which is right.** Possibilities: a slip in the printed table or the data it was built from; a fault in our code that affects exactly these n; a transcription error on our side (the 1996 preprint and the 2022 retypeset version print the same numbers, so OCR is unlikely). This is the first question put to the reader in the paper (Section 8).

## Table 2.4 and beyond

Table 2.4 lists n = 10, 20, …, 1000. Our DFS reaches n = 1000 in about two seconds.

In [4]:
import time
t = time.time(); d = bl.odd_fixed_points_dfs(1000); print('n=1000 in %.1fs' % (time.time() - t))
bad = [n for n in sorted(bl.TABLE_2_4) if d[n] != bl.TABLE_2_4[n]]
print('Table 2.4: %d values, differences at n = %s' % (len(bl.TABLE_2_4), bad))
print({n: d[n] for n in (100, 200, 500, 1000)})

n=1000 in 2.1s
Table 2.4: 100 values, differences at n = [70]
{100: 90, 200: 404, 500: 1150, 1000: 2308}


## −1 and 1/3

Bernstein–Lagarias conjecture that Φ has exactly two odd fixed points (−1 and 1/3). At every level the truncations of both are among the odd fixed points; the other odd residues at each level die at deeper levels, which is why the counts grow while the conjecture says two survive. Counting is all this does; it is not evidence for or against the conjecture.

In [5]:
N = 60
for name, x in (('-1', (1 << N) - 1), ('1/3', pow(3, -1, 1 << N))):
    print(name, 'mod 2^%d =' % N, x, ' Q_N(x) == x:', bl.parity_vector(x, N) == x)

-1 mod 2^60 = 1152921504606846975  Q_N(x) == x: True
1/3 mod 2^60 = 768614336404564651  Q_N(x) == x: True
